<a href="https://colab.research.google.com/github/weagan/Share-PEFT/blob/main/Thorough_test_continual_LoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, AutoModel
from datasets import load_dataset
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import numpy as np

## 1. Standard LoRA (Stage 1)
class SimpleLoRALinear(nn.Module):
    def __init__(self, weight, rank=16):
        super().__init__()
        self.out_features, self.in_features = weight.shape
        self.register_buffer("weight", weight.clone())
        # Initialize lora_A and lora_B on the same device as the weight
        self.lora_A = nn.Parameter(torch.randn(rank, self.in_features, device=weight.device) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(self.out_features, rank, device=weight.device))
        self.scaling = 0.05

    def forward(self, x):
        delta_w = self.lora_B @ self.lora_A
        return x @ (self.weight + delta_w * self.scaling).T

## 2. Subspace LoRA (Stage 2 & 3)
class SubspaceLoRALinear(nn.Module):
    def __init__(self, weight, rank=16):
        super().__init__()
        self.out_features, self.in_features = weight.shape
        self.register_buffer("weight", weight.clone())
        self.rank = rank
        # Initialize basis and coefficients on the same device as the weight
        self.register_buffer("basis", torch.zeros(rank, self.out_features * self.in_features, device=weight.device))
        self.coefficients = nn.Parameter(torch.zeros(rank, device=weight.device))
        self.scaling = 0.05

    def forward(self, x):
        delta_w = (self.coefficients @ self.basis).view(self.out_features, self.in_features)
        return x @ (self.weight + delta_w * self.scaling).T

## 3. Manager
class ContinualSubspaceManager:
    def __init__(self, model_name="distilbert-base-uncased", rank=16):
        # Determine the primary device (cuda:0 if any GPU, else cpu)
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

        # Load model and move to primary device
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
        self.model.to(self.device)

        # Load base encoder model for embedding extraction
        self.base_encoder = AutoModel.from_pretrained(model_name)
        self.base_encoder.to(self.device)

        # Check for multiple GPUs and wrap with DataParallel if available
        self.num_gpus = torch.cuda.device_count()
        if self.num_gpus > 1:
            print(f"Using {self.num_gpus} GPUs with DataParallel.")
            self.model = nn.DataParallel(self.model)
            self.base_encoder = nn.DataParallel(self.base_encoder)

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.rank = rank
        self.task_coefficients = {}

        # Freeze the base model parameters.
        # If DataParallel is used, freeze the parameters of the underlying module.
        base_model_for_freezing = self.model.module if isinstance(self.model, nn.DataParallel) else self.model
        for param in base_model_for_freezing.parameters():
            param.requires_grad = False

        base_encoder_for_freezing = self.base_encoder.module if isinstance(self.base_encoder, nn.DataParallel) else self.base_encoder
        for param in base_encoder_for_freezing.parameters():
            param.requires_grad = False

    def _get_target_module_and_attr(self, name, model_obj):
        """Helper to get the actual parent module and attribute name to modify."""
        parts = name.split('.')
        current_obj = model_obj
        for part in parts[:-1]:
            current_obj = getattr(current_obj, part)
        return current_obj, parts[-1]

    def inject_standard_lora(self):
        # Work on the base model, whether it's wrapped by DataParallel or not
        target_model = self.model.module if isinstance(self.model, nn.DataParallel) else self.model
        for name, module in list(target_model.named_modules()): # Use list to allow modification during iteration
            if any(tgt_name in name for tgt_name in ["attention.out_lin", "attention.v_lin"]):
                parent, attr_name = self._get_target_module_and_attr(name, target_model)
                setattr(parent, attr_name, SimpleLoRALinear(module.weight, self.rank))

    def convert_to_subspace(self):
        target_model = self.model.module if isinstance(self.model, nn.DataParallel) else self.model
        with torch.no_grad():
            for name, module in list(target_model.named_modules()):
                if isinstance(module, SimpleLoRALinear):
                    delta_w = module.lora_B @ module.lora_A
                    # Perform SVD on CPU to be safer with memory, then move results back to device
                    U, S, Vh = torch.linalg.svd(delta_w.cpu(), full_matrices=False)
                    new_module = SubspaceLoRALinear(module.weight, self.rank) # Initialized on correct device
                    for r in range(self.rank):
                        # Ensure outer product is on the correct device.
                        new_module.basis[r] = torch.outer(U[:, r], Vh[r, :]).flatten().to(module.weight.device)
                    parent, attr_name = self._get_target_module_and_attr(name, target_model)
                    setattr(parent, attr_name, new_module)

    def train_task(self, task_name, loader, is_subspace=True, epochs=5):
        print(f"\n🔥 Training {task_name} ({'Subspace' if is_subspace else 'Standard LoRA'})")
        keyword = "coefficients" if is_subspace else "lora_"
        # Collect parameters from the actual trainable module
        trainable_model = self.model.module if isinstance(self.model, nn.DataParallel) else self.model
        params = [p for n, p in trainable_model.named_parameters() if keyword in n]
        optimizer = optim.AdamW(params, lr=1e-3)

        self.model.train()
        for epoch in range(epochs):
            for batch in tqdm(loader, leave=False):
                optimizer.zero_grad()
                labels = batch.pop("labels").to(self.device)
                inputs = {k: v.to(self.device) for k, v in batch.items()}
                # The model (DataParallel or not) takes inputs and labels
                outputs = self.model(**inputs, labels=labels)
                outputs.loss.backward()
                optimizer.step()

        if is_subspace:
            # Store coefficients from the actual module
            trainable_model = self.model.module if isinstance(self.model, nn.DataParallel) else self.model
            self.task_coefficients[task_name] = {n: p.clone().detach() for n, p in trainable_model.named_parameters() if "coefficients" in n}

    def evaluate_task(self, loader):
        self.model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for batch in loader:
                labels = batch.pop("labels").to(self.device)
                inputs = {k: v.to(self.device) for k, v in batch.items()}
                outputs = self.model(**inputs)
                correct += (outputs.logits.argmax(-1) == labels).sum().item()
                total += labels.size(0)
        return correct / total

    def load_task_weights(self, task_name):
        if task_name in self.task_coefficients:
            with torch.no_grad():
                # Load weights into the actual module
                target_model = self.model.module if isinstance(self.model, nn.DataParallel) else self.model
                for n, p in target_model.named_parameters():
                    if n in self.task_coefficients[task_name]:
                        p.copy_(self.task_coefficients[task_name][n])

## 4. Execution Logic
def get_loader(task, tokenizer):
    ds = load_dataset("glue", task)
    def tokenize_fn(ex):
        t = (ex["sentence"],) if "sentence" in ex else (ex["sentence1"], ex["sentence2"])
        # FIX: Changed padding=False to padding=True to ensure uniform tensor lengths when return_tensors='pt'
        res = tokenizer(*t, truncation=True, padding=True, return_tensors='pt')
        res["labels"] = ex["label"]
        return res
    tok = ds.map(tokenize_fn, batched=True, remove_columns=ds["train"].column_names)

    # The dataset mapping will result in dictionary values as lists/tensors. Ensure 'labels' is correctly handled.
    # For the DataLoader, we need to ensure the collator works correctly with potentially batched tensors from map.
    # DataCollatorWithPadding handles this for input_ids, attention_mask. We just need to make sure labels are correct.

    # Manual handling to convert lists of tensors to actual tensors for DataLoader if needed, or ensure the collator does it.
    # For this structure, DataCollatorWithPadding will handle input_ids and attention_mask. Labels should be handled separately.
    # Let's verify what `tok` looks like. `map` generally returns lists of values, which then need to be converted to tensors.
    # We'll rely on the default collator for labels and DataCollatorWithPadding for inputs.

    return DataLoader(tok["train"], batch_size=16, shuffle=True, collate_fn=DataCollatorWithPadding(tokenizer)), \
           DataLoader(tok["validation"], batch_size=16, collate_fn=DataCollatorWithPadding(tokenizer))

def extract_embeddings(model, loader, device):
    model.eval()
    all_embeddings = []
    all_labels = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Extracting Embeddings", leave=False):
            labels = batch.pop("labels") # Keep labels on CPU for now
            inputs = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**inputs, output_hidden_states=True)
            # Get the [CLS] token embedding (first token's last hidden state)
            embeddings = outputs.last_hidden_state[:, 0, :].cpu().detach()
            all_embeddings.append(embeddings)
            all_labels.append(labels)
    return torch.cat(all_embeddings), torch.cat(all_labels)

def train_linear_probe(embeddings, labels, num_labels, device, epochs=10, lr=1e-3):
    input_dim = embeddings.shape[1]
    linear_probe = nn.Linear(input_dim, num_labels).to(device)
    optimizer = optim.AdamW(linear_probe.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    dataset = TensorDataset(embeddings.to(device), labels.to(device))
    dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

    linear_probe.train()
    for epoch in tqdm(range(epochs), desc="Training Linear Probe", leave=False):
        for batch_embeddings, batch_labels in dataloader:
            optimizer.zero_grad()
            outputs = linear_probe(batch_embeddings)
            loss = criterion(outputs, batch_labels)
            loss.backward()
            optimizer.step()
    return linear_probe

def evaluate_linear_probe(linear_probe, embeddings, labels, device):
    linear_probe.eval()
    correct, total = 0, 0
    dataset = TensorDataset(embeddings.to(device), labels.to(device))
    dataloader = DataLoader(dataset, batch_size=32, shuffle=False)

    with torch.no_grad():
        for batch_embeddings, batch_labels in dataloader:
            outputs = linear_probe(batch_embeddings)
            predictions = outputs.argmax(dim=-1)
            correct += (predictions == batch_labels).sum().item()
            total += batch_labels.size(0)
    return correct / total

manager = ContinualSubspaceManager()
tasks = ["cola", "mrpc", "sst2"]
loaders = {t: get_loader(t, manager.tokenizer) for t in tasks}

baseline_accuracies = {}

# Ensure the model is in evaluation mode and on the correct device
# The manager's evaluate_task already sets model.eval()

print("Evaluating base model on validation sets...")
for task in tasks:
    _, val_loader = loaders[task]
    acc = manager.evaluate_task(val_loader)
    baseline_accuracies[task] = acc
    print(f"  {task} baseline accuracy: {acc:.3f}")

print("Baseline accuracies:", baseline_accuracies)
print(manager.device)


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

Evaluating base model on validation sets...
  cola baseline accuracy: 0.309
  mrpc baseline accuracy: 0.321
  sst2 baseline accuracy: 0.489
Baseline accuracies: {'cola': 0.3087248322147651, 'mrpc': 0.32107843137254904, 'sst2': 0.48853211009174313}
cuda:0


In [ ]:
import numpy as np

def count_parameters(model):
    trainable_model = model.module if isinstance(model, nn.DataParallel) else model
    total_params = sum(p.numel() for p in trainable_model.parameters())

    classification_head_params = 0
    classification_head_names = []
    for name, p in trainable_model.named_parameters():
        if "pre_classifier" in name or "classifier" in name:
            classification_head_params += p.numel()
            classification_head_names.append(name)

    print(f"Total parameters in the model: {total_params:,}")
    print(f"Parameters in classification head ({', '.join(set([n.split('.')[0] for n in classification_head_names]))}): {classification_head_params:,}")

    if total_params > 0:
        percent_classification = (classification_head_params / total_params) * 100
        print(f"Percentage of parameters in classification head: {percent_classification:.2f}%")
    else:
        print("No parameters found in the model.")

count_parameters(manager.model)

Total parameters in the model: 66,955,010
Parameters in classification head (classifier, pre_classifier): 592,130
Percentage of parameters in classification head: 0.88%


In [ ]:
linear_probe_accuracies = {}
task_num_labels = {"cola": 2, "mrpc": 2, "sst2": 2} # All these tasks are binary classification

print("\n--- LINEAR PROBING BASELINE ACCURACIES ---")
for task in tasks:
    train_loader, val_loader = loaders[task]
    num_labels = task_num_labels[task]

    print(f"  Extracting embeddings for {task} (train)...")
    train_embeddings, train_labels = extract_embeddings(manager.base_encoder, train_loader, manager.device)
    print(f"  Extracting embeddings for {task} (validation)...")
    val_embeddings, val_labels = extract_embeddings(manager.base_encoder, val_loader, manager.device)

    print(f"  Training linear probe for {task}...")
    linear_probe = train_linear_probe(train_embeddings, train_labels, num_labels, manager.device)

    acc = evaluate_linear_probe(linear_probe, val_embeddings, val_labels, manager.device)
    linear_probe_accuracies[task] = acc
    print(f"  {task} linear probe accuracy: {acc:.3f}")

print("\nLinear Probe Baseline accuracies:", linear_probe_accuracies)


--- LINEAR PROBING BASELINE ACCURACIES ---
  Extracting embeddings for cola (train)...


  Extracting embeddings for cola (validation)...


  Training linear probe for cola...


  cola linear probe accuracy: 0.749
  Extracting embeddings for mrpc (train)...


  Extracting embeddings for mrpc (validation)...


  Training linear probe for mrpc...


  mrpc linear probe accuracy: 0.691
  Extracting embeddings for sst2 (train)...


  Extracting embeddings for sst2 (validation)...


  Training linear probe for sst2...


  sst2 linear probe accuracy: 0.842

Linear Probe Baseline accuracies: {'cola': 0.7488015340364333, 'mrpc': 0.6911764705882353, 'sst2': 0.841743119266055}


In [ ]:
forgetting_table = np.zeros((3, 3))

# Stage 1: CoLA Warmup
manager.inject_standard_lora()
manager.train_task("cola", loaders["cola"][0], is_subspace=False)
manager.convert_to_subspace() # Creates the shared Basis

# Stage 2 & 3: Train others in subspace and evaluate
for i, task in enumerate(tasks):
    if i > 0: # CoLA is already trained
        manager.train_task(task, loaders[task][0], is_subspace=True)

    for j in range(i + 1):
        prev_task = tasks[j]
        manager.load_task_weights(prev_task)
        acc = manager.evaluate_task(loaders[prev_task][1])
        forgetting_table[i, j] = acc

## 5. Result Display
print("\n--- BASELINE ACCURACIES ---")
for task_name, acc in baseline_accuracies.items():
    print(f"  {task_name} baseline accuracy: {acc:.3f}")

print("\n--- LINEAR PROBE BASELINE ACCURACIES ---")
for task_name, acc in linear_probe_accuracies.items():
    print(f"  {task_name} linear probe accuracy: {acc:.3f}")

print("\n--- FORGETTING TABLE (Accuracy) ---")
print(f"{'State':<12} | {'CoLA':<6} | {'MRPC':<6} | {'SST2':<6}")
for i, row in enumerate(forgetting_table):
    row_str = " | ".join([f"{v:.3f}" if v > 0 else "-----" for v in row])
    print(f"After {tasks[i]:<7} | {row_str}")



🔥 Training cola (Standard LoRA)



🔥 Training mrpc (Subspace)



🔥 Training sst2 (Subspace)



--- BASELINE ACCURACIES ---
  cola baseline accuracy: 0.309
  mrpc baseline accuracy: 0.321
  sst2 baseline accuracy: 0.489

--- LINEAR PROBE BASELINE ACCURACIES ---
  cola linear probe accuracy: 0.749
  mrpc linear probe accuracy: 0.691
  sst2 linear probe accuracy: 0.842

--- FORGETTING TABLE (Accuracy) ---
State        | CoLA   | MRPC   | SST2  
After cola    | 0.319 | ----- | -----
After mrpc    | 0.342 | 0.380 | -----
After sst2    | 0.677 | 0.380 | 0.525
